In [ ]:
# Cell 1: install dependencies (run once)
# If you already have transformers/datasets/accelerate/evaluate, you can skip.
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
# Cell 2: imports & environment
import os
os.environ["WANDB_DISABLED"] = "true"   # disable Weights & Biases logging

import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)
import evaluate

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)


torch: 2.9.0+cu126
cuda available: True


In [ ]:
# Cell 3: config — change model_name to "bert-large-uncased" if you want Large
model_name = "bert-base-uncased"   # or "bert-large-uncased"
output_dir = "./bert_imdb_finetune"
max_length = 128

# For BERT-Base (default)
per_device_train_batch_size = 8
per_device_eval_batch_size = 8
num_train_epochs = 2
learning_rate = 2e-5

# If you switch to BERT-Large, uncomment these adjustments:
if "large" in model_name:
    # reduce batch size, use gradient accumulation / fp16 to fit GPU memory
    per_device_train_batch_size = 2
    per_device_eval_batch_size = 2
    # You can enable fp16 in TrainingArguments below
    print("Using BERT Large: adjusted batch sizes for memory.")

In [ ]:
# Cell 4: load IMDB dataset (train/test)
dataset = load_dataset("imdb")
train_ds = dataset["train"]   # 25k
test_ds = dataset["test"]     # 25k

# Sample a subset of the dataset to speed up training
sample_percentage = 0.1 # Use 10% of the dataset for faster training
sample_size_train = int(sample_percentage * len(train_ds))
sample_size_test = int(sample_percentage * len(test_ds))

train_ds = train_ds.shuffle(seed=42).select(range(sample_size_train))
test_ds = test_ds.shuffle(seed=42).select(range(sample_size_test))

print(f"Sampled Train size: {len(train_ds)} (original: {len(dataset['train'])})")
print(f"Sampled Test size: {len(test_ds)} (original: {len(dataset['test'])})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Sampled Train size: 2500 (original: 25000)
Sampled Test size: 2500 (original: 25000)


In [ ]:
# Cell 5: tokenizer and tokenization function
tokenizer = BertTokenizerFast.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",   # ensures fixed input shape; optionally use DataCollatorWithPadding instead
        max_length=max_length
    )

# tokenize (batched)
train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])

# rename label -> labels for HF Trainer
train_ds = train_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

# set torch format
train_ds.set_format("torch")
test_ds.set_format("torch")
print("Columns:", train_ds.column_names)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# Cell 6: model and data collator
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

# If you used padding="max_length" above, DataCollator is optional.
# But it's good practice to use dynamic padding for variable-length batches:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Cell 7: evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
    }

In [ ]:
# Cell 8: TrainingArguments and Trainer
training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    logging_steps=500,
    save_total_limit=2,
    fp16=("large" in model_name) and torch.cuda.is_available(),  # use fp16 for large if GPU supports it
    gradient_accumulation_steps=1 if "base" in model_name else 2, # bump accumulation for large if needed
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-4005699065.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Cell 9: train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.368442,0.847600,0.852611,0.829819,0.876691
2,0.375500,0.500234,0.855600,0.861100,0.833830,0.890215


TrainOutput(global_step=626, training_loss=0.3578039723844193, metrics={'train_runtime': 183.2634, 'train_samples_per_second': 27.283, 'train_steps_per_second': 3.416, 'total_flos': 328888819200000.0, 'train_loss': 0.3578039723844193, 'epoch': 2.0})

In [ ]:
# Cell 10: save
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print("Saved to", output_dir)

Saved to ./bert_imdb_finetune


In [ ]:
# Cell 11: evaluate / test metrics
results = trainer.evaluate(eval_dataset=test_ds)
print("Test results:", results)

Test results: {'eval_loss': 0.5002337694168091, 'eval_accuracy': 0.8556, 'eval_f1': 0.8611004232397076, 'eval_precision': 0.8338301043219076, 'eval_recall': 0.8902147971360382, 'eval_runtime': 16.99, 'eval_samples_per_second': 147.145, 'eval_steps_per_second': 18.423, 'epoch': 2.0}


In [ ]:
# Cell 12: inference - single & batch
from transformers import pipeline

# easy pipeline (loads saved model automatically if you pass path)
sentiment = pipeline("text-classification", model=output_dir, tokenizer=output_dir, device=0 if torch.cuda.is_available() else -1)

examples = [
    "The movie was engaging, well-acted, and emotionally powerful.",
    "This was a complete waste of time. Terrible acting."
]

preds = sentiment(examples, truncation=True, max_length=max_length)
for ex, p in zip(examples, preds):
    print(ex)
    print("->", p)

Device set to use cuda:0


The movie was engaging, well-acted, and emotionally powerful.
-> {'label': 'LABEL_1', 'score': 0.9936654567718506}
This was a complete waste of time. Terrible acting.
-> {'label': 'LABEL_0', 'score': 0.9912959337234497}


In [ ]:
# Cell 13: get CLS vector for sentences (useful for downstream tasks)
def get_cls_embeddings(texts, batch_size=32):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
            outputs = model.bert(**enc)  # returns BaseModelOutputWithPoolingAndCrossAttentions
            # `pooler_output` is the pooled CLS vector (after dense + tanh)
            cls = outputs.pooler_output.cpu().numpy()
            embeddings.append(cls)
    return np.vstack(embeddings)

sample_texts = ["A great movie!", "I did not like the film."]
cls_embs = get_cls_embeddings(sample_texts)
print("CLS embeddings shape:", cls_embs.shape)  # (2, hidden_size)


CLS embeddings shape: (2, 768)
